
# ETL Data Architecture in Spark Streaming
### Part A &#183; Architecture of Spark Streaming &#8212; Real-Time Streaming with Kafka &amp; Spark


Phần này trả lời trọn vẹn yêu cầu **ETL Data Architecture in Spark Streaming** của đề bài (Part A &#8211; 30 marks):

1. **Giải thích tầm quan trọng của ETL** (Extract &#8211; Transform &#8211; Load) trong xử lý dữ liệu liên tục.
2. **Spark Streaming hỗ trợ ETL như thế nào** qua: *schema enforcement*, *stateful transformations*, *fault tolerance (Write-Ahead Log + checkpointing)*, và *end-to-end exactly-once semantics*.
3. **Tóm tắt + trích dẫn** một paper về tối ưu ETL streaming (yêu cầu *literature* trong rubric).

Mọi ví dụ code dùng đúng **schema Retail Rocket** (`timestamp, visitorid, event, itemid, transactionid`) để khớp với pipeline Part B của nhóm.



## 1. ETL là gì &amp; tại sao quan trọng trong xử lý liên tục

**ETL** là quy trình chuẩn hóa dữ liệu gồm 3 giai đoạn:

- **Extract (Trích xuất):** lấy dữ liệu thô từ nguồn &#8212; trong hệ streaming là một *dòng sự kiện vô hạn* (Kafka topic, log, clickstream).
- **Transform (Biến đổi):** parse, làm sạch, ép kiểu, lọc, làm giàu (enrich), tính toán tổng hợp (aggregate) theo thời gian.
- **Load (Nạp):** ghi kết quả đã chuẩn hóa ra nơi lưu trữ / phục vụ (data lake, warehouse, dashboard, API).



<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 260" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="260" fill="#ffffff"/>
  <text x="430" y="28" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">ETL trong xử lý dữ liệu liên tục (continuous / streaming)</text>

  <!-- EXTRACT -->
  <rect x="30" y="70" width="210" height="120" rx="10" fill="#dbeafe" stroke="#1e40af" stroke-width="1.5"/>
  <text x="135" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#1e3a8a">EXTRACT</text>
  <text x="135" y="122" text-anchor="middle" font-size="12" fill="#1e3a8a">Đọc dòng sự kiện vô hạn</text>
  <text x="135" y="140" text-anchor="middle" font-size="12" fill="#1e3a8a">Kafka / File / Socket</text>
  <text x="135" y="162" text-anchor="middle" font-size="11" fill="#334155">spark.readStream</text>

  <!-- TRANSFORM -->
  <rect x="325" y="70" width="210" height="120" rx="10" fill="#dcfce7" stroke="#166534" stroke-width="1.5"/>
  <text x="430" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#166534">TRANSFORM</text>
  <text x="430" y="122" text-anchor="middle" font-size="12" fill="#166534">Parse JSON, ép kiểu, làm sạch</text>
  <text x="430" y="140" text-anchor="middle" font-size="12" fill="#166534">Window + Watermark + Agg</text>
  <text x="430" y="162" text-anchor="middle" font-size="11" fill="#334155">select / withColumn / groupBy</text>

  <!-- LOAD -->
  <rect x="620" y="70" width="210" height="120" rx="10" fill="#fef3c7" stroke="#92400e" stroke-width="1.5"/>
  <text x="725" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#92400e">LOAD</text>
  <text x="725" y="122" text-anchor="middle" font-size="12" fill="#92400e">Ghi ra sink (bền vững)</text>
  <text x="725" y="140" text-anchor="middle" font-size="12" fill="#92400e">Parquet / Delta / Console</text>
  <text x="725" y="162" text-anchor="middle" font-size="11" fill="#334155">writeStream + checkpoint</text>

  <!-- arrows -->
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#475569"/>
    </marker>
  </defs>
  <line x1="245" y1="130" x2="320" y2="130" stroke="#475569" stroke-width="2.5" marker-end="url(#ah)"/>
  <line x1="540" y1="130" x2="615" y2="130" stroke="#475569" stroke-width="2.5" marker-end="url(#ah)"/>

  <text x="430" y="228" text-anchor="middle" font-size="12" fill="#64748b">Luồng chạy liên tục, không ngừng &#8212; mỗi micro-batch lặp lại E → T → L trên dữ liệu mới</text>
  <text x="430" y="247" text-anchor="middle" font-size="11" fill="#94a3b8">Kiểm soát đúng đắn bằng checkpoint (WAL + state) → exactly-once</text>
</svg>
</div>



### 1.1. Vì sao phải làm ETL *liên tục* thay vì *theo lô* (batch)?

Đề bài nêu rõ: trong khi batch (ví dụ Airflow) xử lý dữ liệu tĩnh theo lịch, các bài toán hiện đại (**fraud detection, real-time metrics, live personalization**) đòi hỏi xử lý sự kiện *dưới giây*. Nếu chờ gộp theo lô, giá trị của dữ liệu (ví dụ ý định mua hàng của user) đã biến mất trước khi kịp phản hồi.

<table>
<thead><tr><th>Tiêu chí</th><th>Batch ETL</th><th>Streaming ETL</th></tr></thead>
<tbody>
<tr><td><b>Đơn vị dữ liệu</b></td><td>Bounded &#8211; file/bảng cố định</td><td>Unbounded &#8211; dòng sự kiện vô hạn</td></tr>
<tr><td><b>Độ trễ (latency)</b></td><td>phút &#8211; giờ</td><td>mili-giây &#8211; giây</td></tr>
<tr><td><b>Kích hoạt</b></td><td>theo lịch (cron / Airflow)</td><td>liên tục (micro-batch / continuous)</td></tr>
<tr><td><b>Trạng thái (state)</b></td><td>stateless mỗi lần chạy</td><td>state tồn tại qua các batch (window, aggregation)</td></tr>
<tr><td><b>Dữ liệu đến muộn</b></td><td>chạy lại toàn bộ lô</td><td>watermark + xử lý late data</td></tr>
<tr><td><b>Ví dụ</b></td><td>báo cáo cuối ngày</td><td>fraud detection, live recommendation</td></tr>
</tbody></table>

### 1.2. ETL vs **ELT** &#8212; khác nhau chỗ nào?

<table>
<thead><tr><th>Tiêu chí</th><th>ETL (Extract &#8594; Transform &#8594; Load)</th><th>ELT (Extract &#8594; Load &#8594; Transform)</th></tr></thead>
<tbody>
<tr><td><b>Nơi transform</b></td><td>Trong engine xử lý (Spark) <i>trước khi</i> lưu</td><td>Trong kho đích (warehouse/lakehouse) <i>sau khi</i> lưu dữ liệu thô</td></tr>
<tr><td><b>Thời điểm làm sạch</b></td><td>Ngay trên đường đi (on-the-fly)</td><td>Sau, khi truy vấn</td></tr>
<tr><td><b>Phù hợp</b></td><td>Cần data quality/schema nghiêm ngặt, latency thấp</td><td>Dữ liệu thô khối lượng lớn, compute đàn hồi ở warehouse</td></tr>
<tr><td><b>Trong đồ án này</b></td><td>&#10003; Spark Structured Streaming transform ngay khi đọc từ Kafka</td><td>Thường là batch trong warehouse</td></tr>
</tbody></table>

> **Kết luận:** Với bài toán *Real-Time E-Commerce Session Tracker* của nhóm, **streaming ETL** là lựa chọn đúng: phải phát hiện ý định mua (high intent) **trước khi phiên kết thúc**, điều mà batch không làm kịp.



## 2. Spark Structured Streaming như một ETL engine

Structured Streaming coi dòng dữ liệu như một **bảng vô hạn (unbounded table)** liên tục có dòng mới được chèn vào. Điểm mạnh then chốt cho ETL: **cùng một đoạn code DataFrame chạy được cho cả batch lẫn streaming** &#8212; chỉ khác `read` ↔ `readStream` và `write` ↔ `writeStream`. Engine tự động *incrementalize* (biến truy vấn thành kế hoạch thực thi tăng dần).

**Ánh xạ ETL &#8594; Spark API:**

<table>
<thead><tr><th>Giai đoạn ETL</th><th>Vai trò</th><th>Spark Structured Streaming API</th></tr></thead>
<tbody>
<tr><td><b>Extract</b></td><td>Đọc nguồn unbounded</td><td><code>spark.readStream.format("kafka"/"json"/"rate").schema(...)</code></td></tr>
<tr><td><b>Transform (stateless)</b></td><td>Parse / lọc / ép kiểu</td><td><code>select, filter, withColumn, from_json, cast</code></td></tr>
<tr><td><b>Transform (stateful)</b></td><td>Tổng hợp theo thời gian</td><td><code>withWatermark, groupBy(window(...)), join, dropDuplicates</code></td></tr>
<tr><td><b>Load</b></td><td>Ghi ra sink có đảm bảo</td><td><code>writeStream.outputMode(...).option("checkpointLocation",...).start()</code></td></tr>
</tbody></table>

**3 chế độ xuất (output mode)** &#8212; quyết định cái gì được ghi ra sink mỗi batch:

| Output mode | Ghi gì ra sink | Dùng khi |
|---|---|---|
| **append** | chỉ dòng mới (bất biến) | ETL không aggregate, hoặc aggregate có watermark |
| **update** | các dòng kết quả vừa thay đổi | aggregate cần cập nhật live |
| **complete** | toàn bộ bảng kết quả | aggregate nhỏ, dashboard tổng |

Dưới đây khởi động Spark &#8212; **chạy cell này trước** (kernel *Python (spark_env)*).


In [1]:
# ===== KHỞI ĐỘNG SPARK (chạy 1 lần, đầu buổi) =====
import os, sys, json, time, random, shutil, glob

# Ép Spark worker chạy đúng Python của kernel này (tránh lệch phiên bản Python)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F          # F.* = các hàm cột: col, sum, window, from_json...
from pyspark.sql.types import (                 # các kiểu dữ liệu để khai báo schema
    StructType, StructField, LongType, IntegerType, StringType, TimestampType)

spark = (
    SparkSession.builder                              # bắt đầu dựng session
    .appName("PartA-ETL-Architecture")                # tên app (hiện trong log/Spark UI)
    .master("local[2]")                               # chạy cục bộ 2 luồng, không cần cụm
    .config("spark.sql.shuffle.partitions", "4")      # số partition khi shuffle -> để nhỏ cho demo nhẹ
    .config("spark.ui.enabled", "false")              # tắt web UI cho gọn
    .config("spark.ui.showConsoleProgress", "false")  # tắt thanh tiến độ -> output sạch
    .getOrCreate()                                    # dùng session đang có, hoặc tạo mới
)
spark.sparkContext.setLogLevel("ERROR")               # chỉ in lỗi, ẩn log WARN/INFO rườm rà

# In ra để xác nhận Spark đã sẵn sàng
print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Python driver :", sys.version.split()[0])

26/10/26 23:08:56 WARN Utils: Your hostname, MacBook-Pro-ca-Tun-Phong.local resolves to a loopback address: 127.0.0.1; using 192.168.1.140 instead (on interface en0)
26/10/26 23:08:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/26 23:08:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.9
Master        : local[2]
Python driver : 3.10.20


### 2.1. Dữ liệu vào từ đâu? Kafka khác gì nguồn khác

Để demo mà **không cần dựng Kafka**, ở đây mình sinh sẵn các file JSON đúng schema Retail Rocket vào một thư mục, rồi đọc bằng **file streaming source** (mỗi file mới = một micro-batch). Sang Part B chỉ cần đổi nguồn sang Kafka, phần phía sau giữ nguyên.

Điểm cần nhớ là **dữ liệu lấy ra có định dạng khác nhau tùy nguồn:**

| Nguồn | Spark trả về cái gì | Phải làm thêm |
|---|---|---|
| **File (JSON/CSV)** | Các cột đã đúng kiểu theo schema mình khai | Dùng được ngay |
| **Kafka** | Một bản ghi "bì thư" cố định: `key`, `value`, `topic`, `partition`, `offset`, `timestamp` — trong đó `value` là **nhị phân (binary)**, chưa đọc được | Phải `CAST(value AS STRING)` rồi `from_json(...)` để bóc thành cột |

Nói ngắn gọn: **file đưa cho mình dữ liệu đã mở sẵn; còn Kafka đưa một phong bì gồm metadata cộng phần nội dung `value` vẫn dán kín — mình phải tự mở nó ra (ép về string rồi `from_json`)**. Bù lại, Kafka cho biết thêm `offset`/`partition`, chính là thứ giúp đọc lại được khi lỗi — phục vụ exactly-once ở mục 7.

In [2]:
# ===== TẠO DỮ LIỆU GIẢ LẬP (thay Kafka trong demo) =====
# Sinh file JSON đúng schema Retail Rocket để bước Extract có cái mà đọc.
random.seed(42)                      # cố định ngẫu nhiên -> chạy lại ra kết quả y hệt

BASE = "/tmp/retailrocket_demo"
LANDING = f"{BASE}/landing"          # thư mục "bãi đáp" = nguồn cho file streaming source
shutil.rmtree(BASE, ignore_errors=True)    # xóa dữ liệu cũ nếu chạy lại
os.makedirs(LANDING, exist_ok=True)

EPOCH0 = 1433221332117               # mốc thời gian gốc (~2015-06-02), giống dataset thật
EVENTS = [("view", 80), ("addtocart", 15), ("transaction", 5)]   # loại event + tỉ lệ xuất hiện

def make_event(i):                   # tạo 1 sự kiện ngẫu nhiên đúng cấu trúc Retail Rocket
    # random.choices: bốc loại event theo trọng số (view hay gặp nhất, transaction hiếm)
    ev = random.choices([e for e, _ in EVENTS], weights=[w for _, w in EVENTS])[0]
    return {
        "timestamp": EPOCH0 + i * 15000,                  # mỗi event cách nhau 15 giây
        "visitorid": random.randint(1, 12),               # id khách (1..12 cho dễ nhìn)
        "event": ev,
        "itemid": random.randint(1000, 1008),             # id sản phẩm
        "transactionid": (random.randint(1, 9999) if ev == "transaction" else None),  # chỉ có khi mua
    }

# Ghi 3 "lô" file (mỗi lô 60 dòng JSON) -> mô phỏng dữ liệu đến dần theo thời gian
idx = 0
for b in range(3):
    rows = [json.dumps(make_event(idx + k)) for k in range(60)]   # 60 event -> 60 dòng JSON
    with open(f"{LANDING}/events_{b:03d}.json", "w") as f:
        f.write("\n".join(rows))
    idx += 60

print("Đã tạo", len(os.listdir(LANDING)), "file tại", LANDING)
print("\nVí dụ 2 bản ghi thô:")
for line in open(sorted(glob.glob(f"{LANDING}/*.json"))[0]).read().splitlines()[:2]:
    print(" ", line)

Đã tạo 3 file tại /tmp/retailrocket_demo/landing

Ví dụ 2 bản ghi thô:
  {"timestamp": 1433221332117, "visitorid": 1, "event": "view", "itemid": 1004, "transactionid": null}
  {"timestamp": 1433221347117, "visitorid": 3, "event": "view", "itemid": 1001, "transactionid": null}


## 3. Extract — đọc dòng dữ liệu streaming

Bước Extract gồm mấy việc, làm lần lượt:

1. **Chọn nguồn và format** — `spark.readStream.format("kafka" / "json" / ...)`.
2. **Khai báo schema** — bắt buộc, vì streaming không tự đoán schema được (lý do ở mục 4).
3. **Thêm option đọc** — ví dụ `maxFilesPerTrigger` cho file, hay `subscribe` + `startingOffsets` cho Kafka.
4. **Gọi `.load()`** (hoặc `.json(path)`) → nhận về một **streaming DataFrame**.
5. *(Riêng Kafka)* tách phần nội dung: `CAST(value AS STRING)` rồi `from_json` theo schema.

Lưu ý: streaming DataFrame là **lazy** — nó mới chỉ mô tả "sẽ đọc gì", chưa thật sự chạy cho tới khi có `writeStream...start()` ở bước Load. Cell dưới làm đúng các bước 1–4 với file source.

In [3]:
# ===== EXTRACT: đọc thư mục thành một dòng dữ liệu (stream) =====
# schema = "hợp đồng dữ liệu": khai trước đủ 5 cột và kiểu của chúng
schema = StructType([
    StructField("timestamp",     LongType(),    True),   # epoch mili-giây
    StructField("visitorid",     IntegerType(), True),
    StructField("event",         StringType(),  True),
    StructField("itemid",        IntegerType(), True),
    StructField("transactionid", IntegerType(), True),
])

raw_stream = (
    spark.readStream                     # readStream = đọc ở chế độ streaming (nguồn vô hạn)
    .schema(schema)                      # áp schema (bắt buộc: streaming không tự đoán được)
    .option("maxFilesPerTrigger", 1)     # mỗi lần chỉ nuốt 1 file -> mô phỏng từng micro-batch
    .json(LANDING)                       # đọc JSON trong thư mục landing
)

print("Đây có phải streaming DataFrame? ->", raw_stream.isStreaming)   # True = đúng là luồng
raw_stream.printSchema()                 # in schema đã áp

Đây có phải streaming DataFrame? -> True
root
 |-- timestamp: long (nullable = true)
 |-- visitorid: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- transactionid: integer (nullable = true)




## 4. Schema enforcement &#8212; "hợp đồng dữ liệu" (data contract)

Trong streaming, dữ liệu đến liên tục và **không thể dừng lại để sửa**. Vì vậy *phải* áp schema tường minh ngay từ đầu:

- **Spark không infer schema cho stream** (vì không thể quét toàn bộ dòng vô hạn) → ta phải khai báo.
- Schema đóng vai trò **hợp đồng dữ liệu** giữa producer và consumer: sai kiểu, thiếu trường → phát hiện ngay thay vì lầm lẫn về sau.
- Dữ liệu từ Kafka là **byte/JSON string** → dùng `from_json(value, schema)` để bóc thành cột có kiểu.
- Bản ghi hỏng được đưa vào cột `_corrupt_record` (hoặc null) thay vì làm sập job.

<table>
<thead><tr><th></th><th>Schema infer (tự đoán)</th><th>Schema enforcement (khai báo)</th></tr></thead>
<tbody>
<tr><td><b>Streaming</b></td><td>Không khả dụng với dòng vô hạn</td><td>Bắt buộc &#10003;</td></tr>
<tr><td><b>An toàn kiểu</b></td><td>Dễ trôi kiểu giữa các batch</td><td>Ổn định, nhất quán</td></tr>
<tr><td><b>Hiệu năng</b></td><td>Tốn 1 lượt quét để đoán</td><td>Không tốn lượt quét</td></tr>
</tbody></table>

Dưới đây minh họa `from_json` trên một cột chuỗi JSON (đúng như `value` lấy từ Kafka), và cho thấy bản ghi hỏng trở thành `null` thay vì làm hỏng pipeline.


In [4]:
# ===== SCHEMA ENFORCEMENT: from_json biến chuỗi JSON -> cột có kiểu =====
# 3 dòng "value" dạng chuỗi (giống hệt value lấy từ Kafka); dòng thứ 3 CỐ TÌNH hỏng
sample = spark.createDataFrame([
    ('{"timestamp":1433221332117,"visitorid":257597,"event":"addtocart","itemid":355908,"transactionid":null}',),
    ('{"timestamp":1433221347117,"visitorid":99,"event":"view","itemid":1001,"transactionid":null}',),
    ('{ chuỗi JSON HỎNG - thiếu dấu ngoặc',),     # bản ghi lỗi -> xem Spark xử lý thế nào
], ["value"])

# from_json(value, schema): đọc chuỗi JSON theo schema -> 1 struct "d"; rồi d.* tách thành các cột
parsed = sample.select(F.from_json("value", schema).alias("d")).select("d.*")
parsed.show(truncate=False)
print("Bản ghi hỏng -> toàn bộ cột = null (pipeline KHÔNG sập).")

+-------------+---------+---------+------+-------------+
|timestamp    |visitorid|event    |itemid|transactionid|
+-------------+---------+---------+------+-------------+
|1433221332117|257597   |addtocart|355908|NULL         |
|1433221347117|99       |view     |1001  |NULL         |
|NULL         |NULL     |NULL     |NULL  |NULL         |
+-------------+---------+---------+------+-------------+

Bản ghi hỏng -> toàn bộ cột = null (pipeline KHÔNG sập).



## 5. Transform &#8212; stateless vs stateful

Đây là trái tim của ETL. Spark chia 2 nhóm phép biến đổi:

<table>
<thead><tr><th></th><th>Stateless</th><th>Stateful</th></tr></thead>
<tbody>
<tr><td><b>Định nghĩa</b></td><td>Mỗi bản ghi xử lý độc lập</td><td>Cần nhớ trạng thái qua nhiều batch</td></tr>
<tr><td><b>Ví dụ</b></td><td><code>select, filter, withColumn, from_json, cast</code></td><td><code>groupBy(window).agg</code>, <code>join</code>, <code>dropDuplicates</code>, <code>mapGroupsWithState</code></td></tr>
<tr><td><b>Chi phí</b></td><td>Nhẹ, không cần checkpoint state</td><td>Cần lưu state (state store) + watermark để giới hạn bộ nhớ</td></tr>
</tbody></table>

### 5.1. Event-time, Windowing &amp; Watermark

- **Event-time**: thời điểm sự kiện *thực sự xảy ra* (trường `timestamp`), khác với *processing-time* (lúc Spark nhận được). Streaming ETL đúng đắn phải dựa trên event-time.
- **Window**: gom sự kiện theo khoảng thời gian. *Tumbling* (không chồng lấn) vs *sliding* (trượt, có chồng lấn) vs *session*.
- **Watermark** `withWatermark("event_time", "5 minutes")`: báo cho engine &#8220;không chờ dữ liệu trễ quá 5 phút&#8221; → cho phép **giải phóng state cũ** và vẫn xử lý dữ liệu đến muộn hợp lý. Đề bài Part B yêu cầu chính xác: *window 10 phút, watermark 5 phút*.



<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 300" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="300" fill="#ffffff"/>
  <text x="430" y="26" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">Sliding window 10&#8217; (trượt 5&#8217;) + Watermark 5&#8217;</text>

  <!-- time axis -->
  <line x1="40" y1="250" x2="820" y2="250" stroke="#334155" stroke-width="2" marker-end="url(#ah2)"/>
  <defs>
    <marker id="ah2" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#334155"/>
    </marker>
  </defs>
  <text x="820" y="272" text-anchor="end" font-size="12" fill="#334155">event time</text>
  <!-- ticks -->
  <g font-size="11" fill="#64748b" text-anchor="middle">
    <line x1="120" y1="245" x2="120" y2="255" stroke="#64748b"/><text x="120" y="270">10:00</text>
    <line x1="280" y1="245" x2="280" y2="255" stroke="#64748b"/><text x="280" y="270">10:05</text>
    <line x1="440" y1="245" x2="440" y2="255" stroke="#64748b"/><text x="440" y="270">10:10</text>
    <line x1="600" y1="245" x2="600" y2="255" stroke="#64748b"/><text x="600" y="270">10:15</text>
    <line x1="760" y1="245" x2="760" y2="255" stroke="#64748b"/><text x="760" y="270">10:20</text>
  </g>

  <!-- windows -->
  <rect x="120" y="70" width="320" height="34" rx="6" fill="#dbeafe" stroke="#1e40af"/>
  <text x="280" y="92" text-anchor="middle" font-size="12" fill="#1e3a8a">W1 [10:00 &#8211; 10:10)</text>
  <rect x="280" y="112" width="320" height="34" rx="6" fill="#dcfce7" stroke="#166534"/>
  <text x="440" y="134" text-anchor="middle" font-size="12" fill="#166534">W2 [10:05 &#8211; 10:15)</text>
  <rect x="440" y="154" width="320" height="34" rx="6" fill="#fef3c7" stroke="#92400e"/>
  <text x="600" y="176" text-anchor="middle" font-size="12" fill="#92400e">W3 [10:10 &#8211; 10:20)</text>

  <!-- watermark line -->
  <line x1="600" y1="50" x2="600" y2="245" stroke="#dc2626" stroke-width="2" stroke-dasharray="6 4"/>
  <text x="608" y="60" font-size="12" font-weight="700" fill="#dc2626">Watermark = max(event_time) &#8722; 5&#8217;</text>

  <!-- late event -->
  <circle cx="330" cy="215" r="7" fill="#16a34a"/>
  <text x="330" y="235" text-anchor="middle" font-size="11" fill="#166534">event trễ &#10003; nhận</text>
  <circle cx="200" cy="215" r="7" fill="#dc2626"/>
  <text x="200" y="235" text-anchor="middle" font-size="11" fill="#b91c1c">quá trễ &#10007; bỏ</text>

  <text x="430" y="292" text-anchor="middle" font-size="11" fill="#94a3b8">Watermark cho phép state của window cũ được giải phóng, đồng thời vẫn chấp nhận dữ liệu đến muộn trong ngưỡng cho phép</text>
</svg>
</div>



### 5.2. Ví dụ Transform theo đúng nghiệp vụ nhóm: **intent score**

Theo đề bài: gán trọng số tương tác **view = 1, addtocart = 5, transaction = 10**, rồi tính điểm ý định (intent score) cộng dồn theo `visitorid` trên cửa sổ thời gian. Đây là bước **Transform stateful** điển hình.


In [5]:
# ===== TRANSFORM phần 1 (STATELESS): làm sạch + gán điểm cho từng event =====
# create_map: bảng tra trọng số theo loại event (view=1, addtocart=5, transaction=10)
weight_map = F.create_map(
    F.lit("view"),        F.lit(1),
    F.lit("addtocart"),   F.lit(5),
    F.lit("transaction"), F.lit(10),
)

clean_stream = (
    raw_stream
    # cast: đổi timestamp từ mili-giây (số) sang kiểu thời gian thật -> cột event_time
    .withColumn("event_time", (F.col("timestamp") / 1000).cast(TimestampType()))
    .withColumn("weight", weight_map[F.col("event")])   # tra trọng số cho mỗi dòng
    .filter(F.col("event").isNotNull())                 # bỏ dòng hỏng (event rỗng)
)

# ===== TRANSFORM phần 2 (STATEFUL): cộng điểm theo cửa sổ thời gian =====
intent_stream = (
    clean_stream
    .withWatermark("event_time", "5 minutes")   # chịu dữ liệu đến trễ tối đa 5' rồi mới chốt cửa sổ
    .groupBy(
        F.window("event_time", "10 minutes", "5 minutes").alias("w"),  # gom theo cửa sổ 10', trượt 5'
        F.col("visitorid"),                                            # và theo từng khách
    )
    .agg(
        F.sum("weight").alias("intent_score"),      # tổng điểm ý định của khách trong cửa sổ
        F.count("*").alias("num_events"),           # số sự kiện
        # đếm riêng số lần add-to-cart và số lần mua (dùng cho logic gợi ý)
        F.sum(F.when(F.col("event") == "addtocart", 1).otherwise(0)).alias("addtocart_cnt"),
        F.sum(F.when(F.col("event") == "transaction", 1).otherwise(0)).alias("txn_cnt"),
    )
)
print("Streaming? ->", intent_stream.isStreaming, "| đã dựng xong kế hoạch Transform (lazy).")

Streaming? -> True | đã dựng xong kế hoạch Transform (lazy).



Chạy truy vấn stateful trên toàn bộ dữ liệu hiện có (`trigger availableNow` → xử lý hết rồi dừng), ghi vào **memory sink** để xem kết quả như một bảng:


In [6]:
# ===== CHẠY query stateful để xem kết quả intent =====
q_intent = (
    intent_stream.writeStream                 # writeStream = bắt đầu CHẠY luồng (trước đó mới là kế hoạch)
    .format("memory").queryName("intent_tbl") # ghi vào bảng trong RAM tên "intent_tbl" để query lại
    .outputMode("complete")                   # complete = mỗi lần xuất toàn bộ bảng tổng hợp
    .trigger(availableNow=True)               # xử lý hết dữ liệu đang có rồi DỪNG (tiện demo)
    .start()
)
q_intent.awaitTermination()                   # chờ query chạy xong

# Query bảng vừa tạo để lấy top điểm intent
print(">>> Điểm intent cao nhất theo (visitor, window):")
spark.sql("""
    SELECT visitorid,
           date_format(w.start, 'HH:mm') AS w_start,   -- giờ bắt đầu cửa sổ
           date_format(w.end,   'HH:mm') AS w_end,      -- giờ kết thúc cửa sổ
           intent_score, num_events, addtocart_cnt, txn_cnt
    FROM intent_tbl
    ORDER BY intent_score DESC
    LIMIT 8
""").show(truncate=False)

>>> Điểm intent cao nhất theo (visitor, window):
+---------+-------+-----+------------+----------+-------------+-------+
|visitorid|w_start|w_end|intent_score|num_events|addtocart_cnt|txn_cnt|
+---------+-------+-----+------------+----------+-------------+-------+
|9        |12:15  |12:25|32          |10        |1            |2      |
|9        |12:10  |12:20|25          |7         |0            |2      |
|2        |12:20  |12:30|18          |6         |3            |0      |
|12       |12:30  |12:40|18          |5         |1            |1      |
|12       |12:25  |12:35|17          |4         |1            |1      |
|2        |12:35  |12:45|17          |5         |3            |0      |
|4        |12:15  |12:25|17          |8         |0            |1      |
|10       |12:05  |12:15|16          |4         |3            |0      |
+---------+-------+-----+------------+----------+-------------+-------+




### 5.3. Kích hoạt khuyến nghị (recommendation trigger)

Đề bài: lọc **high-intent** (intent score &#8805; 5) và đặc biệt là *addtocart nhưng chưa transaction* → kích hoạt gợi ý ngay. Đây là điểm giao giữa ETL (Transform) và Recommender (phần của nhóm khác) &#8212; ETL chuẩn bị đúng &#8220;tín hiệu&#8221; để hệ gợi ý dùng.


In [7]:
# ===== LOGIC GỢI Ý: lọc khách ý định cao để kích hoạt recommendation =====
reco_df = spark.sql("""
    SELECT visitorid,
           date_format(w.start, 'HH:mm') AS w_start,
           intent_score, addtocart_cnt, txn_cnt,
           -- đã thêm giỏ nhưng chưa mua -> ưu tiên gợi ý ngay
           CASE WHEN addtocart_cnt > 0 AND txn_cnt = 0
                THEN 'GỢI Ý NGAY (đã add-to-cart, chưa mua)'
                ELSE 'theo dõi tiếp' END AS action
    FROM intent_tbl
    WHERE intent_score >= 5        -- chỉ lấy khách có ý định cao (>=5 điểm)
    ORDER BY intent_score DESC
    LIMIT 10
""")
reco_df.show(truncate=False)

+---------+-------+------------+-------------+-------+-------------------------------------+
|visitorid|w_start|intent_score|addtocart_cnt|txn_cnt|action                               |
+---------+-------+------------+-------------+-------+-------------------------------------+
|9        |12:15  |32          |1            |2      |theo dõi tiếp                        |
|9        |12:10  |25          |0            |2      |theo dõi tiếp                        |
|2        |12:20  |18          |3            |0      |GỢI Ý NGAY (đã add-to-cart, chưa mua)|
|12       |12:30  |18          |1            |1      |theo dõi tiếp                        |
|12       |12:25  |17          |1            |1      |theo dõi tiếp                        |
|2        |12:35  |17          |3            |0      |GỢI Ý NGAY (đã add-to-cart, chưa mua)|
|4        |12:15  |17          |0            |1      |theo dõi tiếp                        |
|3        |12:05  |16          |0            |1      |theo dõi tiếp   


## 6. Fault tolerance &#8212; Write-Ahead Log (WAL) + Checkpointing

Câu hỏi sống còn của streaming ETL: *nếu job crash giữa chừng thì sao?* Spark trả lời bằng **checkpoint** &#8212; bắt buộc khai báo `checkpointLocation` cho mọi query production. Thư mục checkpoint gồm:

| Thư mục | Vai trò |
|---|---|
| **offsets/** | **Write-Ahead Log**: ghi *trước* khoảng offset sẽ xử lý cho mỗi batch (ghi trước khi làm → &#8220;write-ahead&#8221;) |
| **commits/** | Đánh dấu batch đã hoàn tất và sink đã commit |
| **state/** | State store (có phiên bản theo batch) cho các phép stateful |
| **sources/** | Metadata nguồn (ví dụ file source: danh sách file đã đọc) |
| **metadata** | ID của query |

**Cơ chế phục hồi:** khi restart, Spark đọc `offsets/` để biết batch nào dở dang, so với `commits/` để biết đã commit tới đâu, khôi phục `state/` đúng phiên bản, rồi **xử lý lại đúng batch còn thiếu**.

Minh họa bằng một sink có checkpoint, sau đó *soi* thư mục checkpoint:


In [8]:
# ===== LOAD: ghi ra Parquet + log mỗi micro-batch + bật checkpoint =====
CKPT    = f"{BASE}/_checkpoint_curated"    # nơi lưu checkpoint (offsets=WAL + commits + state)
CURATED = f"{BASE}/curated_parquet"        # nơi lưu dữ liệu đã làm sạch (Parquet)
shutil.rmtree(CKPT, ignore_errors=True)
shutil.rmtree(CURATED, ignore_errors=True)

# foreachBatch: hàm được gọi 1 lần cho MỖI micro-batch -> chỗ vừa ghi dữ liệu vừa log
def process_batch(batch_df, batch_id):
    t0 = time.time()
    n = batch_df.count()                                 # đếm số bản ghi của batch này
    (batch_df.write.mode("append")                       # ghi nối thêm (không ghi đè)
             .partitionBy("event")                       # chia thư mục theo loại event
             .parquet(CURATED))
    # log: id batch, số record, thời gian xử lý (đáp ứng yêu cầu structured logging của đề)
    print(f"[BATCH {batch_id}] records={n:<4d} elapsed={time.time()-t0:0.2f}s "
          f"-> ghi Parquet (partitioned by event)")

q_curated = (
    clean_stream.select("event_time", "visitorid", "event", "itemid", "weight")  # chọn cột cần ghi
    .writeStream
    .foreachBatch(process_batch)                 # gắn hàm xử lý từng batch ở trên
    .option("checkpointLocation", CKPT)          # BẬT checkpoint -> chịu lỗi + exactly-once
    .trigger(availableNow=True)
    .start()
)
q_curated.awaitTermination()
print("\nĐã nạp xong vào Parquet curated.")

[BATCH 0] records=60   elapsed=0.80s -> ghi Parquet (partitioned by event)


[BATCH 1] records=60   elapsed=0.15s -> ghi Parquet (partitioned by event)


[BATCH 2] records=60   elapsed=0.14s -> ghi Parquet (partitioned by event)

Đã nạp xong vào Parquet curated.


In [9]:
# ===== SOI CHECKPOINT: nhìn tận mắt WAL + commit log =====
print("Nội dung thư mục checkpoint:")
for name in sorted(os.listdir(CKPT)):        # liệt kê thư mục con của checkpoint
    print("  ├─", name)                      # thường có: offsets, commits, state, sources, metadata

def first_file(sub):                         # lấy file đầu tiên trong một thư mục con
    p = os.path.join(CKPT, sub)
    fs = [f for f in sorted(os.listdir(p)) if not f.startswith(".")] if os.path.isdir(p) else []
    return os.path.join(p, fs[0]) if fs else None

off = first_file("offsets")                  # file trong offsets/ CHÍNH LÀ WAL
if off:
    print(f"\n--- Nội dung offset log batch đầu tiên ({os.path.basename(off)}) = WAL ---")
    print(open(off).read().strip())          # in ra -> thấy Spark ghi offset TRƯỚC khi xử lý
print("\ncommits/:", os.listdir(os.path.join(CKPT, "commits")))   # batch nào đã commit xong

Nội dung thư mục checkpoint:
  ├─ .metadata.crc
  ├─ commits
  ├─ metadata
  ├─ offsets
  ├─ sources

--- Nội dung offset log batch đầu tiên (0) = WAL ---
v1
{"batchWatermarkMs":0,"batchTimestampMs":1793030944483,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.optimizer.pruneFiltersCanPruneStreamingSubplan":"false","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
{"logOffset":0}

commits/: ['.2.crc', '.0.crc', '.1.crc', '0', '1', '2']


In [10]:
# ===== KIỂM CHỨNG LOAD: đọc lại dữ liệu đã ghi =====
curated = spark.read.parquet(CURATED)        # đọc lại Parquet (chế độ batch thường)
print("Tổng bản ghi curated:", curated.count())
curated.groupBy("event").count().orderBy(F.desc("count")).show()   # đếm theo loại event
print("Phân vùng trên đĩa (partitionBy event):")
for d in sorted(glob.glob(f"{CURATED}/event=*")):    # mỗi loại event = 1 thư mục partition
    print("  ", os.path.basename(d))

Tổng bản ghi curated: 180
+-----------+-----+
|      event|count|
+-----------+-----+
|       view|  151|
|  addtocart|   22|
|transaction|    7|
+-----------+-----+

Phân vùng trên đĩa (partitionBy event):
   event=addtocart
   event=transaction
   event=view


## 7. End-to-end exactly-once semantics

Câu hỏi đặt ra: nếu job lỗi rồi chạy lại, dữ liệu có bị **tính hai lần** hoặc **mất** không? **Exactly-once** nghĩa là mỗi sự kiện ảnh hưởng tới kết quả **đúng một lần**, kể cả khi có lỗi và phải chạy lại.

Có ba mức đảm bảo:

<table>
<thead><tr><th>Mức</th><th>Ý nghĩa</th><th>Rủi ro</th></tr></thead>
<tbody>
<tr><td><b>At-most-once</b></td><td>mỗi record xử lý tối đa 1 lần</td><td>có thể <b>mất</b> dữ liệu</td></tr>
<tr><td><b>At-least-once</b></td><td>mỗi record ít nhất 1 lần</td><td>có thể <b>trùng lặp</b></td></tr>
<tr><td><b>Exactly-once</b></td><td>đúng 1 lần về hiệu lực</td><td>cần 3 thứ phối hợp dưới đây</td></tr>
</tbody></table>

**Nói đơn giản, Spark làm được exactly-once nhờ 3 thứ phối hợp:**

1. **Nguồn đọc lại được** — Kafka/file nhớ vị trí đã đọc (offset), nên chạy lại biết đọc tiếp từ đâu.
2. **Checkpoint ghi lại tiến độ** — Spark ghi "sắp xử lý tới đâu" *trước khi* làm (đây chính là WAL), và lưu cả trạng thái (state).
3. **Sink ghi không trùng** — đoạn nào đã ghi xong thì lần chạy lại bỏ qua, không ghi lần hai.

Gộp lại: **nhớ đã làm tới đâu + đọc lại được + ghi không trùng ⇒ chạy lại vẫn ra kết quả như làm đúng một lần.**


<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 280" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="280" fill="#ffffff"/>
  <text x="430" y="26" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">Cơ chế end-to-end exactly-once</text>
  <defs>
    <marker id="ah3" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#475569"/>
    </marker>
  </defs>

  <!-- source -->
  <rect x="30" y="80" width="170" height="90" rx="10" fill="#dbeafe" stroke="#1e40af"/>
  <text x="115" y="108" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">1. Replayable source</text>
  <text x="115" y="130" text-anchor="middle" font-size="11" fill="#1e3a8a">Kafka offsets / file log</text>
  <text x="115" y="148" text-anchor="middle" font-size="11" fill="#1e3a8a">đọc lại được khi fail</text>

  <!-- engine + checkpoint -->
  <rect x="250" y="70" width="200" height="110" rx="10" fill="#dcfce7" stroke="#166534"/>
  <text x="350" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#166534">2. Engine + Checkpoint</text>
  <text x="350" y="118" text-anchor="middle" font-size="11" fill="#166534">offset log (WAL) ghi trước</text>
  <text x="350" y="136" text-anchor="middle" font-size="11" fill="#166534">state store có phiên bản</text>
  <text x="350" y="154" text-anchor="middle" font-size="11" fill="#166534">commit log sau khi xong</text>

  <!-- sink -->
  <rect x="500" y="80" width="170" height="90" rx="10" fill="#fef3c7" stroke="#92400e"/>
  <text x="585" y="108" text-anchor="middle" font-size="13" font-weight="700" fill="#92400e">3. Idempotent sink</text>
  <text x="585" y="130" text-anchor="middle" font-size="11" fill="#92400e">manifest / transaction</text>
  <text x="585" y="148" text-anchor="middle" font-size="11" fill="#92400e">bỏ batch đã commit</text>

  <!-- result -->
  <rect x="712" y="80" width="118" height="90" rx="10" fill="#ede9fe" stroke="#6d28d9"/>
  <text x="771" y="120" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">Kết quả</text>
  <text x="771" y="142" text-anchor="middle" font-size="12" fill="#5b21b6">đúng 1 lần</text>

  <line x1="202" y1="125" x2="246" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>
  <line x1="452" y1="125" x2="496" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>
  <line x1="672" y1="125" x2="708" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>

  <text x="430" y="212" text-anchor="middle" font-size="12" fill="#334155">Khi job crash giữa chừng: Spark đọc lại <tspan font-weight="700">offset log</tspan> để biết batch dở dang, phục hồi <tspan font-weight="700">state</tspan> từ checkpoint,</text>
  <text x="430" y="232" text-anchor="middle" font-size="12" fill="#334155">xử lý lại đúng batch đó; sink dựa vào <tspan font-weight="700">commit log</tspan> để không ghi trùng → hiệu lực <tspan font-weight="700">exactly-once</tspan>.</text>
</svg>
</div>



> **Lưu ý cho Part B:** cặp **Kafka source + Parquet/Delta sink + `checkpointLocation`** cho nhóm end-to-end exactly-once miễn phí. Đó là lý do mọi `writeStream` trong pipeline đều phải có checkpoint riêng. Console sink *không* đảm bảo exactly-once (chỉ dùng để demo).

Bằng chứng thực nghiệm đơn giản cho tính **idempotent**: chạy lại cùng query với cùng checkpoint → **không** sinh thêm bản ghi trùng (vì các batch đã nằm trong commit log):


In [11]:
# ===== CHỨNG MINH IDEMPOTENT: chạy lại KHÔNG ghi trùng =====
count_before = spark.read.parquet(CURATED).count()   # số bản ghi trước khi chạy lại

def process_batch_again(batch_df, batch_id):
    print(f"[RERUN BATCH {batch_id}] records={batch_df.count()}")
    batch_df.write.mode("append").partitionBy("event").parquet(CURATED)

q_rerun = (
    clean_stream.select("event_time", "visitorid", "event", "itemid", "weight")
    .writeStream.foreachBatch(process_batch_again)
    .option("checkpointLocation", CKPT)          # DÙNG LẠI checkpoint cũ -> Spark biết đã xử lý hết
    .trigger(availableNow=True).start()
)
q_rerun.awaitTermination()

count_after = spark.read.parquet(CURATED).count()    # số bản ghi sau khi chạy lại
print(f"\nTrước rerun: {count_before} | Sau rerun: {count_after}")
print("=> Bằng nhau: offset log cho biết mọi file đã xử lý, KHÔNG ghi trùng (idempotent).")


Trước rerun: 180 | Sau rerun: 180
=> Bằng nhau: offset log cho biết mọi file đã xử lý, KHÔNG ghi trùng (idempotent).



## 8. Tổng hợp: kiến trúc ETL theo tầng (Medallion)

Một cách tổ chức streaming ETL phổ biến là **Medallion architecture** &#8212; chia pipeline thành 3 lớp chất lượng tăng dần:

<table>
<thead><tr><th>Lớp</th><th>Vai trò</th><th>Trong đồ án</th></tr></thead>
<tbody>
<tr><td><b> Bronze</b></td><td>Dữ liệu thô, giữ nguyên (raw)</td><td>Event JSON từ Kafka, lưu nguyên bản</td></tr>
<tr><td><b> Silver</b></td><td>Đã làm sạch, ép kiểu, chuẩn hóa</td><td>Parse timestamp, gán weight, lọc hợp lệ</td></tr>
<tr><td><b> Gold</b></td><td>Tổng hợp phục vụ nghiệp vụ</td><td>Intent score theo window → recommendation</td></tr>
</tbody></table>

Cả 3 lớp chính là E → T → L mà ta đã dựng ở trên. Bảng dưới tổng kết toàn bộ luồng bằng một foreachBatch log gọn:


In [12]:
# ===== TỔNG KẾT: log 3 lớp Bronze/Silver/Gold cho mỗi micro-batch =====
SUMMARY_CKPT = f"{BASE}/_checkpoint_summary"
shutil.rmtree(SUMMARY_CKPT, ignore_errors=True)

def medallion_log(batch_df, batch_id):
    bronze = batch_df.count()                                       # Bronze: dữ liệu thô
    silver = batch_df.filter(F.col("event").isNotNull()).count()    # Silver: đã lọc sạch
    # Gold: số khách có tổng điểm >= 5 (ý định cao)
    gold   = (batch_df.groupBy("visitorid").agg(F.sum("weight").alias("s"))
                      .filter(F.col("s") >= 5).count())
    print(f"[micro-batch {batch_id}] Bronze(raw)={bronze:<4d} "
          f"Silver(clean)={silver:<4d} Gold(high-intent visitors)={gold}")

q_sum = (
    clean_stream
    .writeStream.foreachBatch(medallion_log)     # mỗi batch -> in log 3 lớp
    .option("checkpointLocation", SUMMARY_CKPT)
    .trigger(availableNow=True).start()
)
q_sum.awaitTermination()
print("\nMỗi micro-batch đều đi qua đủ Extract -> Transform -> Load, có logging + checkpoint.")

[micro-batch 0] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=8


[micro-batch 1] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=8


[micro-batch 2] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=9

Mỗi micro-batch đều đi qua đủ Extract -> Transform -> Load, có logging + checkpoint.


## 9. Literature — tóm tắt một nghiên cứu về tối ưu ETL streaming

> **Trích dẫn (APA):** Armbrust, M., et al. (2018). *Structured Streaming: A Declarative API for Real-Time Applications in Apache Spark.* SIGMOD '18, 601–613. ACM. https://doi.org/10.1145/3183713.3190664

**Bài báo giải quyết vấn đề gì?** Cách viết streaming kiểu cũ (DStream) bắt lập trình viên tự lo chuyện chia nhỏ theo lô, tự xử lý dữ liệu đến trễ, và tự chống trùng/mất dữ liệu — rất dễ sai.

**Ý tưởng chính:**
- Coi dòng dữ liệu như **một bảng cứ dài thêm mãi**, và cho viết truy vấn **y như làm với dữ liệu tĩnh** — cùng một đoạn code chạy được cho cả batch lẫn streaming.
- **Spark tự lo phần khó**: tự biến truy vấn đó thành cách xử lý từng phần nhỏ, nên người viết **không phải tự code logic phức tạp**.
- Tự động đảm bảo **đúng một lần (exactly-once)** bằng checkpoint và ghi không trùng — đúng những thứ mình vừa xem ở mục 6 và 7.
- **Nhanh**: theo bài báo, nhanh khoảng **gấp đôi Apache Flink** trong thử nghiệm của họ.

**Vì sao liên quan đến nhóm:** nó chứng minh một hệ ETL streaming có thể **vừa dễ viết, vừa chạy nhanh, vừa đảm bảo đúng** — chính là nền tảng cho pipeline Retail Rocket của nhóm.


## 10. Tổng kết &amp; liên hệ Part B

<table>
<thead><tr><th>Yêu cầu đề bài (ETL)</th><th>Đã trả lời ở mục</th></tr></thead>
<tbody>
<tr><td>Tầm quan trọng của ETL trong xử lý liên tục</td><td>1 (so sánh batch/streaming, ETL/ELT)</td></tr>
<tr><td>Schema enforcement</td><td>4 (<code>from_json</code>, data contract)</td></tr>
<tr><td>Stateful transformations</td><td>5 (window + watermark + intent score)</td></tr>
<tr><td>Fault tolerance (WAL + checkpointing)</td><td>6 (soi thư mục checkpoint)</td></tr>
<tr><td>End-to-end exactly-once</td><td>7 (3 điều kiện + kiểm chứng idempotent)</td></tr>
<tr><td>Literature (cited paper)</td><td>9 (Armbrust et al., SIGMOD 2018)</td></tr>
</tbody></table>

**Cầu nối sang Part B:** toàn bộ code Transform (parse `from_json`, `event_time`, weight, `window`+`withWatermark`, intent score, `foreachBatch` logging, Parquet sink + `checkpointLocation`) đúng yêu cầu Task 3 `streaming_etl.py`. Khi chuyển sang Part B, chỉ cần thay **Extract: file source → Kafka source** (`spark.readStream.format("kafka")`), phần còn lại giữ nguyên.

### Tài liệu tham khảo
1. Armbrust, M., et al. (2018). *Structured Streaming: A Declarative API for Real-Time Applications in Apache Spark.* SIGMOD &#8217;18, 601&#8211;613. https://doi.org/10.1145/3183713.3190664
2. Apache Spark. *Structured Streaming Programming Guide.* https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html
3. Zaharia, M., et al. (2013). *Discretized Streams: Fault-Tolerant Streaming Computation at Scale.* SOSP &#8217;13.


In [13]:
# ===== DỌN DẸP: tắt Spark khi trình bày xong (chạy cuối buổi) =====
spark.stop()     # giải phóng tài nguyên; sau lệnh này muốn dùng tiếp phải tạo lại SparkSession
print("Đã dừng SparkSession. Xong phần ETL Data Architecture in Spark Streaming.")

Đã dừng SparkSession. Xong phần ETL Data Architecture in Spark Streaming.
